In [3]:
cd "/content/drive/MyDrive/Courses/AI/masked_attention/llama_like"

/content/drive/MyDrive/Courses/AI/masked_attention/llama_like


In [15]:
# ---------------------------------------------------------------------------
# Generation
# ---------------------------------------------------------------------------
@torch.no_grad()
def generate(
    model: MiniLLaMA,
    tokenizer: BPETokenizer,
    prompt: str,
    max_new_tokens: int = 40,
    strategy: str = "greedy",       # "greedy" | "sampling"
    temperature: float = 1.0,
    top_k: int = 0,                  # 0 = disabled
) -> str:
    """
    Autoregressively generate text from a prompt.

    Parameters
    ----------
    prompt         : starting text
    max_new_tokens : how many new tokens to generate
    strategy       : "greedy" or "sampling"
    temperature    : controls randomness (only used with sampling)
    top_k          : if >0, restrict sampling to top-k logits
    """
    model.eval()
    device = next(model.parameters()).device

    # Encode prompt
    input_ids = tokenizer.encode(prompt, add_special_tokens=True)
    input_ids = torch.tensor([input_ids], dtype=torch.long, device=device)  # (1,T)

    generated = input_ids.clone()

    for step in range(max_new_tokens):
        # Truncate to max_seq_len if necessary
        context = generated[:, -model.config.max_seq_len:]

        logits, _ = model(context)          # (1, T, vocab_size)
        next_logits = logits[:, -1, :]      # last position: (1, vocab_size)

        if strategy == "greedy":
            next_token = next_logits.argmax(dim=-1, keepdim=True)   # (1,1)

        elif strategy == "sampling":
            # Scale by temperature
            next_logits = next_logits / max(temperature, 1e-6)

            # Optional top-k filtering
            if top_k > 0:
                top_values, _ = torch.topk(next_logits, top_k)
                threshold = top_values[:, -1].unsqueeze(-1)
                next_logits = next_logits.masked_fill(next_logits < threshold,
                                                      float("-inf"))

            probs      = F.softmax(next_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)     # (1,1)

        else:
            raise ValueError(f"Unknown strategy: {strategy}")

        # Stop at EOS
        if next_token.item() == tokenizer.eos_id:
            break

        generated = torch.cat([generated, next_token], dim=1)

    # Decode (skip the initial BOS token added by encode)
    out_ids = generated[0].tolist()
    return tokenizer.decode(out_ids)

In [18]:
# ---------------------------------------------------------------------------
# Load model from checkpoint
# ---------------------------------------------------------------------------
def load_model(ckpt_dir: str = "checkpoints"):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Tokenizer
    tokenizer = BPETokenizer()
    tokenizer.load(f"{ckpt_dir}/tokenizer.json")

    # Model
    ckpt = torch.load(f"{ckpt_dir}/minillama.pt", map_location=device)
    cfg_dict = ckpt["model_config"]
    model_cfg = ModelConfig(**cfg_dict)

    model = MiniLLaMA(model_cfg).to(device)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()
    print(f"Model loaded from {ckpt_dir}  |  vocab_size={tokenizer.vocab_size}")
    return model, tokenizer

In [20]:
model, tokenizer = load_model("checkpoints")
print(generate(model, tokenizer, prompt="Neural networks are inspired", strategy="sampling", temperature=0.8))

[BPE] Tokenizer loaded from checkpoints/tokenizer.json  vocab_size=555
Model loaded from checkpoints  |  vocab_size=555
Neural networks are inspired . computer vision tasks . Recurrent neural networks are designed to handle sequential data such as text and speech . Transformers replaced recurrent networks and became the dominant architecture for language tasks . The transformer uses self - attention to
